# Day 03 – Differential Expression, GSEA & Pathway Stories

We figure out which genes make each cluster unique and peek at the pathways those genes belong to.


### What happens today?
1. Load the annotated data from Day 02 (auto rebuilds if needed).
2. Run Scanpy's `rank_genes_groups` to get marker genes for every cell type.
3. Optionally send the top genes into GSEA/Enrichr for pathway hints.


In [ ]:
# Install the needed libraries once (delete the # to run)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


### Step 1 – Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print('⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.')

try:
    import gseapy as gp
except ImportError:
    gp = None
    print('⚠️ Install gseapy (pip install gseapy) to run the GSEA step.')

try:
    import scvelo as scv
except ImportError:
    scv = None
    print('⚠️ Install scvelo (pip install scvelo) to run the RNA velocity step.')

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


### Step 2 – Load annotated data (or rebuild it)

In [ ]:
SHARED_DIR = Path('..') / 'shared_data'
SHARED_DIR.mkdir(parents=True, exist_ok=True)
day2_file = SHARED_DIR / 'day02_annotated.h5ad'

print('Looking for Day 02 annotated data at', day2_file)

def build_day2_from_scratch():
    """Create the Day 02 dataset (clean + cluster + annotate) on the fly."""
    data = sc.datasets.pbmc3k()
    data.var_names_make_unique()
    data.layers['counts'] = data.X.copy()
    data.obs['source_dataset'] = 'pbmc3k'
    data.raw = data

    sc.pp.filter_cells(data, min_genes=200)
    sc.pp.filter_genes(data, min_cells=3)
    data.var['mt'] = data.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], inplace=True)
    data = data[data.obs['pct_counts_mt'] < 15, :]

    sc.pp.normalize_total(data, target_sum=1e4)
    sc.pp.log1p(data)
    sc.pp.highly_variable_genes(data, n_top_genes=2000, subset=True)
    sc.pp.scale(data, max_value=10)

    sc.tl.pca(data, n_comps=50)
    sc.pp.neighbors(data, n_neighbors=15)
    sc.tl.umap(data)

    sc.tl.leiden(data, resolution=0.5, key_added='leiden')
    marker_map = {
        '0': 'Naive T',
        '1': 'Memory T',
        '2': 'B cell',
        '3': 'NK',
        '4': 'Myeloid',
        '5': 'Plasma',
    }
    data.obs['cell_type'] = data.obs['leiden'].map(marker_map).fillna('Other')
    data.obs['batch'] = 'Batch_A'
    return data

if day2_file.exists():
    adata_from_day2 = sc.read(day2_file)
    print('✅ Loaded annotated data from Day 02 file.')
else:
    print('⚠️ Day 02 file not found. Re-running the quick Day 02 pipeline now...')
    adata_from_day2 = build_day2_from_scratch()


In [ ]:
adata_day3 = adata_from_day2.copy()
print('Working copy shape:', adata_day3.shape)


### Step 3 – Run simple marker-gene analysis

In [ ]:
sc.tl.rank_genes_groups(adata_day3, groupby='cell_type', method='wilcoxon')
sc.pl.rank_genes_groups(adata_day3, n_genes=5, sharey=False)

marker_df = sc.get.rank_genes_groups_df(adata_day3, group=None)
display(marker_df.head())


### Step 4 – Optional: Send top genes to GSEA/Enrichr

In [ ]:
selected_group = adata_day3.obs['cell_type'].unique()[0]
top_genes = (
    marker_df[marker_df['group'] == selected_group]
    .sort_values('scores', ascending=False)
    .head(50)['names']
    .tolist()
)

if gp is None:
    print('Install gseapy (pip install gseapy) to run pathway enrichment.')
else:
    enr = gp.enrichr(
        gene_list=top_genes,
        description=f'{selected_group}_signature',
        gene_sets='GO_Biological_Process_2023',
        outdir=None
    )
    display(enr.results.head())
